# 🤟 SignBridge — Notebook d'Entraînement des Modèles IA

**Pipeline complet :** Chargement données → Exploration → Prétraitement → Random Forest → LSTM → Évaluation → Export

---

## Formats de datasets supportés

| Format | Description | Exemple |
|--------|-------------|--------|
| **CSV Kaggle** | `label, x0,y0,z0, x1,y1,z1, ...` | `sign_mnist.csv` |
| **JSON landmarks** | `[{"label": "A", "landmarks": [63 floats]}]` | `dataset.json` |
| **Images + MediaPipe** | Dossier `images/A/img1.jpg ...` | `asl_dataset/` |
| **NumPy** | `.npy` arrays X(N,63) et y(N,) | `X.npy`, `y.npy` |

---

## Structure des landmarks MediaPipe
```
21 points × 3 coords (x, y, z) = 63 features
Normalisés : poignet = [0,0,0], scale = dist(poignet → MCP majeur)
```

---
## 0. Configuration

In [ ]:
import os, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ─────────────────────────────────────────────
#  CONFIGURATION — À MODIFIER SELON TON DATASET
# ─────────────────────────────────────────────

# Répertoire racine du projet SignBridge
PROJECT_ROOT = Path('..').resolve()

# Dossier de sortie des modèles
MODEL_DIR = PROJECT_ROOT / 'signbridge_model'

# ── Source du dataset (choisir UN seul) ──────
DATASET_SOURCE = 'csv'          # 'csv' | 'json' | 'images' | 'numpy'

# Chemins selon le type choisi
CSV_PATH    = PROJECT_ROOT / 'dataset' / 'landmarks.csv'
JSON_PATH   = PROJECT_ROOT / 'dataset' / 'landmarks.json'
IMAGES_DIR  = PROJECT_ROOT / 'dataset' / 'images'
NUMPY_X     = PROJECT_ROOT / 'dataset' / 'X.npy'
NUMPY_Y     = PROJECT_ROOT / 'dataset' / 'y.npy'

# ── Paramètres entraînement ──────────────────
TEST_SIZE          = 0.20    # 20 % pour le test
RANDOM_STATE       = 42
MIN_SAMPLES_CLASS  = 10      # classes avec < N samples ignorées

# Random Forest
RF_N_ESTIMATORS    = 300
RF_MAX_DEPTH       = None    # None = arbres complets

# LSTM
SEQ_LEN            = 30      # frames par séquence
LSTM_EPOCHS        = 50
LSTM_BATCH_SIZE    = 32
LSTM_PATIENCE      = 10      # early stopping

MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Projet      : {PROJECT_ROOT}')
print(f'Modèles     : {MODEL_DIR}')
print(f'Source data : {DATASET_SOURCE}')

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#C9D1D9',
    'text.color':       '#C9D1D9',
    'xtick.color':      '#8B949E',
    'ytick.color':      '#8B949E',
    'grid.color':       '#21262D',
    'font.family':      'DejaVu Sans',
})
ACCENT = '#007A5E'

print('✓ Imports OK')
print(f'  NumPy      {np.__version__}')
print(f'  Pandas     {pd.__version__}')
import sklearn; print(f'  scikit-learn {sklearn.__version__}')

---
## 2. Chargement du dataset

Exécute la cellule correspondant à ton format de données.

### 2a. Format CSV (Kaggle / export Django)

In [ ]:
def load_csv(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """
    Charge un CSV landmarks.
    Formats acceptés :
      - label, x0,y0,z0, x1,y1,z1, ...  (63 colonnes numériques)
      - label, landmark_0_x, landmark_0_y, landmark_0_z, ...
      - sign,  pixel_0, pixel_1, ...  (colonnes génériques)
    """
    df = pd.read_csv(path)
    print(f'CSV chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes')
    print(f'Colonnes   : {list(df.columns[:6])} ...')

    # Identifier la colonne label
    label_col = None
    for col in ['label', 'sign', 'class', 'gesture', 'Letter']:
        if col in df.columns:
            label_col = col; break
    if label_col is None:
        label_col = df.columns[0]
    print(f'Colonne label : {label_col!r}')

    y_raw = df[label_col].astype(str).str.upper().str.strip().values
    num_cols = [c for c in df.columns if c != label_col]

    # Prendre les 63 premières colonnes numériques
    X_raw = df[num_cols].select_dtypes(include=[np.number]).values[:, :63].astype(np.float32)
    print(f'Features extraits : {X_raw.shape}')
    return X_raw, y_raw

if DATASET_SOURCE == 'csv':
    X_raw, y_raw = load_csv(CSV_PATH)

### 2b. Format JSON

In [ ]:
def load_json(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """
    Formats acceptés :
      [{"label": "A", "landmarks": [63 floats]}, ...]
      [{"sign": "A", "features": [63 floats]}, ...]
    """
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    X_list, y_list = [], []
    for entry in data:
        lbl = (entry.get('label') or entry.get('sign') or
               entry.get('gesture') or '').upper().strip()
        lm  = entry.get('landmarks') or entry.get('features') or []
        if lbl and len(lm) >= 63:
            X_list.append(lm[:63])
            y_list.append(lbl)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list)
    print(f'JSON chargé : {len(X)} samples, {X.shape[1]} features')
    return X, y

if DATASET_SOURCE == 'json':
    X_raw, y_raw = load_json(JSON_PATH)

### 2c. Format Images (extraction MediaPipe)

In [ ]:
def load_images(images_dir: Path) -> tuple[np.ndarray, np.ndarray]:
    """
    Structure attendue :
      images_dir/
        A/  img1.jpg  img2.jpg ...
        B/  img1.jpg ...
        ...
    Extrait les landmarks MediaPipe de chaque image.
    """
    try:
        import cv2
        import mediapipe as mp
    except ImportError:
        raise ImportError('pip install opencv-python mediapipe')

    mp_hands = mp.solutions.hands
    hands    = mp_hands.Hands(static_image_mode=True,
                               max_num_hands=1,
                               min_detection_confidence=0.5)

    X_list, y_list = [], []
    failed = 0

    class_dirs = sorted([d for d in images_dir.iterdir() if d.is_dir()])
    for class_dir in class_dirs:
        label = class_dir.name.upper().strip()
        images = list(class_dir.glob('*.jpg')) + \
                 list(class_dir.glob('*.jpeg')) + \
                 list(class_dir.glob('*.png'))

        for img_path in images:
            img = cv2.imread(str(img_path))
            if img is None:
                failed += 1; continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            res = hands.process(rgb)
            if not res.multi_hand_landmarks:
                failed += 1; continue

            lm   = res.multi_hand_landmarks[0]
            wrist = lm.landmark[0]
            vec  = []
            for pt in lm.landmark:
                vec.extend([pt.x - wrist.x, pt.y - wrist.y, pt.z - wrist.z])
            X_list.append(vec[:63])
            y_list.append(label)

        print(f'  {label:>3} : {len([x for i,x in enumerate(y_list) if x == label])} samples')

    hands.close()
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list)
    print(f'\nTotal : {len(X)} samples ({failed} images ignorées)')
    return X, y

if DATASET_SOURCE == 'images':
    X_raw, y_raw = load_images(IMAGES_DIR)

### 2d. Format NumPy

In [ ]:
if DATASET_SOURCE == 'numpy':
    X_raw = np.load(NUMPY_X).astype(np.float32)
    y_raw = np.load(NUMPY_Y).astype(str)
    print(f'NumPy chargé : X={X_raw.shape}, y={y_raw.shape}')

---
## 3. Exploration des données (EDA)

In [ ]:
print('═' * 50)
print(f'  Samples total     : {len(X_raw):>8,}')
print(f'  Features          : {X_raw.shape[1]:>8}')
print(f'  Classes uniques   : {len(np.unique(y_raw)):>8}')
print(f'  Valeurs NaN       : {np.isnan(X_raw).sum():>8}')
print(f'  Valeurs infinies  : {np.isinf(X_raw).sum():>8}')
print('═' * 50)

# Distribution par classe
unique_classes, counts = np.unique(y_raw, return_counts=True)
df_counts = pd.DataFrame({'classe': unique_classes, 'count': counts}).sort_values('count', ascending=False)
print(f'\n  Min samples/classe : {counts.min()} ({unique_classes[np.argmin(counts)]})')
print(f'  Max samples/classe : {counts.max()} ({unique_classes[np.argmax(counts)]})')
print(f'  Moy samples/classe : {counts.mean():.0f}')

In [ ]:
# Visualisation distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Barplot
ax = axes[0]
bars = ax.bar(df_counts['classe'], df_counts['count'], color=ACCENT, alpha=0.85, width=0.6)
ax.set_title('Nombre de samples par classe', color='#C9D1D9', fontsize=12, pad=12)
ax.set_xlabel('Classe', color='#8B949E')
ax.set_ylabel('Samples', color='#8B949E')
ax.axhline(counts.mean(), color='#FCD116', linestyle='--', linewidth=1, label=f'Moyenne ({counts.mean():.0f})')
ax.legend()
ax.tick_params(axis='x', rotation=45 if len(unique_classes) > 15 else 0, labelsize=9)

# Déséquilibre
ax2 = axes[1]
ratio = counts / counts.max()
colors = [ACCENT if r > 0.5 else '#CE1126' for r in ratio]
ax2.bar(unique_classes, ratio, color=colors, alpha=0.85, width=0.6)
ax2.set_title('Ratio samples / classe max  (rouge = déséquilibré)', color='#C9D1D9', fontsize=12, pad=12)
ax2.set_xlabel('Classe', color='#8B949E')
ax2.set_ylabel('Ratio', color='#8B949E')
ax2.axhline(0.5, color='#FCD116', linestyle='--', linewidth=1)
ax2.tick_params(axis='x', rotation=45 if len(unique_classes) > 15 else 0, labelsize=9)

plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : eda_distribution.png')

In [ ]:
# Statistiques des features
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(X_raw.flatten(), bins=80, color=ACCENT, alpha=0.8, edgecolor='none')
axes[0].set_title('Distribution de toutes les features', color='#C9D1D9')
axes[0].set_xlabel('Valeur')
axes[0].set_ylabel('Fréquence')

feat_std = X_raw.std(axis=0)
axes[1].bar(range(len(feat_std)), feat_std, color=ACCENT, alpha=0.7, width=1.0)
axes[1].set_title('Variance par feature (21 pts × xyz)', color='#C9D1D9')
axes[1].set_xlabel('Feature index')
axes[1].set_ylabel('Écart-type')

# Annotations des coordonnées
for i, label in enumerate(['x', 'y', 'z'] * 21):
    if feat_std[i] == feat_std.max():
        axes[1].annotate(f'{label} (pt {i//3})', xy=(i, feat_std[i]),
                         xytext=(i+2, feat_std[i]*1.05),
                         color='#FCD116', fontsize=8)

plt.tight_layout()
plt.show()

---
## 4. Prétraitement

In [ ]:
def scale_normalize(lm63: np.ndarray) -> np.ndarray:
    """
    Normalise par la distance poignet → MCP majeur (point 9).
    Rend les features invariants à la taille de main et à la distance caméra.
    """
    lm = lm63.reshape(21, 3).astype(np.float32)
    scale = float(np.linalg.norm(lm[9]))
    # Ne normalise pas si déjà normalisé (scale ≈ 1.0)
    if scale > 1e-6 and abs(scale - 1.0) > 0.05:
        lm /= scale
    return lm.flatten()


# ── Nettoyage ────────────────────────────────────────────────────
# 1. Supprimer les NaN / Inf
mask_valid = ~(np.isnan(X_raw).any(axis=1) | np.isinf(X_raw).any(axis=1))
X_clean = X_raw[mask_valid]
y_clean = y_raw[mask_valid]
print(f'Après nettoyage NaN/Inf : {mask_valid.sum():,} / {len(X_raw):,} samples conservés')

# 2. Supprimer les classes avec trop peu de samples
u, c = np.unique(y_clean, return_counts=True)
valid_classes = u[c >= MIN_SAMPLES_CLASS]
mask_cls = np.isin(y_clean, valid_classes)
X_clean = X_clean[mask_cls]
y_clean = y_clean[mask_cls]
removed = u[c < MIN_SAMPLES_CLASS]
if len(removed):
    print(f'Classes retirées (< {MIN_SAMPLES_CLASS} samples) : {list(removed)}')

# 3. Normalisation d'échelle MediaPipe
X_norm = np.array([scale_normalize(x) for x in X_clean], dtype=np.float32)
print(f'Normalisation d\'échelle : OK  shape={X_norm.shape}')

# 4. Encodage des labels
le = LabelEncoder()
y_enc = le.fit_transform(y_clean)
print(f'Classes encodées : {len(le.classes_)} → {list(le.classes_)}')

# 5. Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_enc,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_enc
)
print(f'\nTrain : {len(X_train):,} samples')
print(f'Test  : {len(X_test):,} samples')

In [ ]:
# Augmentation des données (optionnel)
# Activer pour équilibrer les classes sous-représentées

AUGMENT_DATA = False  # Mettre True pour augmenter
N_AUGMENT    = 200    # Samples synthétiques par classe manquante
SIGMA        = 0.026  # Bruit gaussien (2.6 %)

def augment_sample(lm63: np.ndarray, sigma: float = 0.026) -> np.ndarray:
    """Génère une variante augmentée d'un vecteur landmark."""
    lm = lm63.reshape(21, 3).astype(np.float32)
    v  = lm + np.random.randn(21, 3).astype(np.float32) * sigma
    # Mise à l'échelle ±12 %
    v *= (1.0 + np.random.uniform(-0.12, 0.12))
    # Rotation 2D ±15°
    angle = np.random.uniform(-0.26, 0.26)
    c, s  = np.cos(angle), np.sin(angle)
    xr = v[:, 0] * c - v[:, 1] * s
    yr = v[:, 0] * s + v[:, 1] * c
    v[:, 0] = xr; v[:, 1] = yr
    # Translation légère ±4 %
    v[:, :2] += np.random.uniform(-0.04, 0.04, size=(1, 2))
    return scale_normalize(v.flatten())

if AUGMENT_DATA:
    u_tr, c_tr = np.unique(y_train, return_counts=True)
    target = c_tr.max()
    X_aug_list, y_aug_list = [], []
    for cls_idx, cnt in zip(u_tr, c_tr):
        deficit = min(target - cnt, N_AUGMENT)
        if deficit <= 0:
            continue
        src = X_train[y_train == cls_idx]
        for _ in range(deficit):
            base = src[np.random.randint(len(src))]
            X_aug_list.append(augment_sample(base, SIGMA))
            y_aug_list.append(cls_idx)
    if X_aug_list:
        X_train = np.vstack([X_train, np.array(X_aug_list, dtype=np.float32)])
        y_train = np.concatenate([y_train, np.array(y_aug_list)])
        print(f'Augmentation : +{len(X_aug_list)} samples → train={len(X_train):,}')
else:
    print('Augmentation : désactivée (AUGMENT_DATA = False)')

---
## 5. Entraînement — Random Forest

> **Modèle principal** pour les signes statiques (alphabet A–Z, chiffres 0–9).
> Input : vecteur de 63 features (21 landmarks × x,y,z).

In [ ]:
from datetime import datetime
print(f'Début entraînement RF : {datetime.now().strftime("%H:%M:%S")}')
print(f'Paramètres : n_estimators={RF_N_ESTIMATORS}, max_depth={RF_MAX_DEPTH}, n_jobs=-1')
print(f'Train : {len(X_train):,} samples, {len(le.classes_)} classes')

rf = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_split=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)
rf.fit(X_train, y_train)

rf_train_acc = rf.score(X_train, y_train)
rf_test_acc  = rf.score(X_test,  y_test)

print(f'\nFin entraînement : {datetime.now().strftime("%H:%M:%S")}')
print(f'  Accuracy train : {rf_train_acc:.4f}  ({rf_train_acc*100:.2f} %)')
print(f'  Accuracy test  : {rf_test_acc:.4f}  ({rf_test_acc*100:.2f} %)')

In [ ]:
# Validation croisée (5-fold) — plus robuste qu'un seul split
print('Validation croisée 5-fold (peut prendre ~2 min)...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf, X_norm, y_enc, cv=cv, scoring='accuracy', n_jobs=-1)
print(f'  CV scores  : {[f"{s:.4f}" for s in cv_scores]}')
print(f'  CV mean    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Rapport de classification complet
y_pred_rf = rf.predict(X_test)
print('=== Rapport de classification Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_, digits=3))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_rf)
n  = len(le.classes_)
figsize = max(10, n * 0.45)

fig, ax = plt.subplots(figsize=(figsize, figsize * 0.85))
sns.heatmap(
    cm, annot=True if n <= 20 else False, fmt='d',
    cmap='Greens', linewidths=0.5,
    xticklabels=le.classes_, yticklabels=le.classes_,
    ax=ax, cbar_kws={'shrink': 0.7}
)
ax.set_title(f'Matrice de confusion — Random Forest (accuracy={rf_test_acc:.2%})',
             color='#C9D1D9', fontsize=12, pad=14)
ax.set_xlabel('Prédit', color='#8B949E')
ax.set_ylabel('Réel', color='#8B949E')
plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : rf_confusion_matrix.png')

In [ ]:
# Importance des features
importances = rf.feature_importances_
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

# Noms des features
coord_names = ['x', 'y', 'z']
feat_names  = [f'pt{i//3}_{coord_names[i%3]}' for i in range(63)]

fig, ax = plt.subplots(figsize=(12, 5))
colors_imp = [ACCENT if i % 3 == 0 else '#FCD116' if i % 3 == 1 else '#CE1126'
              for i in top_idx]
ax.bar([feat_names[i] for i in top_idx], importances[top_idx],
       color=colors_imp, alpha=0.85)
ax.set_title(f'Top {top_n} features les plus importantes (RF)', color='#C9D1D9', fontsize=12)
ax.set_xlabel('Feature  (x=vert, y=jaune, z=rouge)')
ax.set_ylabel('Importance')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Entraînement — LSTM (signes dynamiques)

> **Modèle secondaire** pour les signes dynamiques (mots, phrases).
> Input : séquence de 30 frames × 63 features.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  SECTION LSTM — nécessite un dataset de SÉQUENCES
#  Format attendu : X_seq de shape (N, 30, 63), y_seq de shape (N,)
#
#  Si tu n'as que des frames statiques (CSV/JSON landmarks),
#  tu peux simuler des séquences par répétition ou fenêtre glissante.
#  Mettre TRAIN_LSTM = False si pas de données séquentielles.
# ─────────────────────────────────────────────────────────────────────

TRAIN_LSTM = False  # Mettre True si tu as des séquences

# Exemple : charger depuis un fichier NPY de séquences
SEQ_NPY_X = PROJECT_ROOT / 'dataset' / 'sequences_X.npy'
SEQ_NPY_Y = PROJECT_ROOT / 'dataset' / 'sequences_y.npy'

if TRAIN_LSTM:
    if SEQ_NPY_X.exists():
        X_seq = np.load(SEQ_NPY_X).astype(np.float32)
        y_seq_raw = np.load(SEQ_NPY_Y).astype(str)
        print(f'Séquences chargées : X={X_seq.shape}, y={y_seq_raw.shape}')
    else:
        # Simulation depuis données statiques (démonstration)
        print('Simulation de séquences depuis données statiques...')
        N = min(len(X_norm), 2000)
        X_seq = np.stack([X_norm[:N]] * SEQ_LEN, axis=1).astype(np.float32)
        y_seq_raw = y_clean[:N]
        print(f'Séquences simulées : {X_seq.shape}')
else:
    print('TRAIN_LSTM = False — section LSTM ignorée')

In [ ]:
if TRAIN_LSTM:
    import tensorflow as tf
    print(f'TensorFlow {tf.__version__}')

    le_seq  = LabelEncoder()
    y_seq   = le_seq.fit_transform(y_seq_raw)
    n_cls   = len(le_seq.classes_)
    y_cat   = tf.keras.utils.to_categorical(y_seq, num_classes=n_cls)

    Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(
        X_seq, y_cat, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(128, return_sequences=True,
                              input_shape=(SEQ_LEN, 63)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.LSTM(64),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(n_cls, activation='softmax'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            patience=LSTM_PATIENCE, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=5, min_lr=1e-5, verbose=1
        ),
    ]

    print(f'Entraînement LSTM — {len(Xtr_s)} séquences, {n_cls} classes...')
    history = model.fit(
        Xtr_s, ytr_s,
        validation_data=(Xte_s, yte_s),
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )

    _, lstm_acc = model.evaluate(Xte_s, yte_s, verbose=0)
    print(f'\nLSTM test accuracy : {lstm_acc:.4f} ({lstm_acc*100:.2f} %)')

In [ ]:
if TRAIN_LSTM:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric, title in zip(
        axes,
        [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
        ['Accuracy', 'Loss']
    ):
        tr_key, va_key = metric
        epochs = range(1, len(history.history[tr_key]) + 1)
        ax.plot(epochs, history.history[tr_key], color=ACCENT, label='Train', linewidth=2)
        ax.plot(epochs, history.history[va_key], color='#FCD116', label='Val', linewidth=2,
                linestyle='--')
        ax.set_title(f'LSTM — {title}', color='#C9D1D9', fontsize=12)
        ax.set_xlabel('Epoch'); ax.set_ylabel(title)
        ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('lstm_training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Figure sauvegardée : lstm_training_curves.png')

---
## 7. Comparaison de modèles (optionnel)

Compare RF avec d'autres classifieurs sur le même jeu de données.

In [ ]:
COMPARE_MODELS = False  # Mettre True pour comparer (prend plusieurs minutes)

if COMPARE_MODELS:
    from sklearn.svm import SVC
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier

    models_to_compare = {
        'Random Forest (300)': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        'SVM (RBF)':           SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE),
        'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'MLP (256-128)':       MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=200, random_state=RANDOM_STATE),
    }

    results_cmp = {}
    for name, clf in models_to_compare.items():
        clf.fit(X_train, y_train)
        acc = clf.score(X_test, y_test)
        results_cmp[name] = acc
        print(f'  {name:<25} : {acc:.4f}  ({acc*100:.2f} %)')

    fig, ax = plt.subplots(figsize=(10, 4))
    names = list(results_cmp.keys())
    accs  = list(results_cmp.values())
    colors_cmp = [ACCENT if a == max(accs) else '#30363D' for a in accs]
    bars = ax.bar(names, accs, color=colors_cmp, alpha=0.9, width=0.5)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, acc + 0.002,
                f'{acc:.3f}', ha='center', va='bottom', fontsize=10, color='#C9D1D9')
    ax.set_ylim(min(accs)*0.95, 1.02)
    ax.set_title('Comparaison des classifieurs', color='#C9D1D9', fontsize=12)
    ax.set_ylabel('Accuracy (test)')
    ax.tick_params(axis='x', rotation=15)
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## 8. Export des modèles

Sauvegarde vers `signbridge_model/` pour utilisation directe par l'application Django.

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────
rf_path = MODEL_DIR / 'rf_model.pkl'
le_path = MODEL_DIR / 'label_encoder.pkl'

joblib.dump(rf, rf_path, compress=3)
joblib.dump(le, le_path)

rf_size = rf_path.stat().st_size / 1024 / 1024
print(f'RF sauvegardé     : {rf_path}  ({rf_size:.1f} MB)')
print(f'Label encoder     : {le_path}')

# ── LSTM ──────────────────────────────────────────────────────────
if TRAIN_LSTM:
    lstm_path = MODEL_DIR / 'lstm_model.h5'
    model.save(str(lstm_path))
    joblib.dump(le_seq, MODEL_DIR / 'label_encoder_seq.pkl')
    lstm_size = lstm_path.stat().st_size / 1024
    print(f'LSTM sauvegardé   : {lstm_path}  ({lstm_size:.0f} KB)')

# ── Métadonnées JSON ──────────────────────────────────────────────
from datetime import datetime
meta = {
    'trained_at'   : datetime.now().isoformat(),
    'rf_accuracy'  : round(float(rf_test_acc), 4),
    'cv_accuracy'  : round(float(cv_scores.mean()), 4),
    'cv_std'       : round(float(cv_scores.std()), 4),
    'n_classes'    : int(len(le.classes_)),
    'classes'      : list(le.classes_),
    'n_samples'    : int(len(X_norm)),
    'n_train'      : int(len(X_train)),
    'n_test'       : int(len(X_test)),
    'dataset_source': DATASET_SOURCE,
    'rf_params'    : {
        'n_estimators': RF_N_ESTIMATORS,
        'max_depth'   : RF_MAX_DEPTH,
    },
}
if TRAIN_LSTM:
    meta['lstm_accuracy'] = round(float(lstm_acc), 4)
    meta['lstm_epochs']   = len(history.history['loss'])

info_path = MODEL_DIR / 'model_info.json'
with open(info_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f'Métadonnées       : {info_path}')
print()
print('=== RÉSUMÉ FINAL ===')
print(f'  Random Forest   : {rf_test_acc:.2%}  (CV={cv_scores.mean():.2%} ± {cv_scores.std():.2%})')
if TRAIN_LSTM:
    print(f'  LSTM            : {lstm_acc:.2%}')
print(f'  Classes         : {len(le.classes_)}')
print(f'  Samples total   : {len(X_norm):,}')

---
## 9. Test d'inférence

Vérifie que les modèles sauvegardés fonctionnent correctement.

In [ ]:
# Recharge les modèles depuis le disque
rf_loaded = joblib.load(MODEL_DIR / 'rf_model.pkl')
le_loaded = joblib.load(MODEL_DIR / 'label_encoder.pkl')
print(f'Modèle rechargé : {len(le_loaded.classes_)} classes')

# Test sur 10 samples aléatoires du jeu de test
idx = np.random.choice(len(X_test), size=10, replace=False)
X_sample = X_test[idx]
y_true   = le_loaded.inverse_transform(y_test[idx])
y_pred   = le_loaded.inverse_transform(rf_loaded.predict(X_sample))
proba    = rf_loaded.predict_proba(X_sample).max(axis=1)

print('\n  Vrai  → Prédit   (confiance)')
print('  ' + '-' * 35)
for true, pred, conf in zip(y_true, y_pred, proba):
    status = '✓' if true == pred else '✗'
    print(f'  {status}  {true:>5} → {pred:<5}   ({conf:.2%})')

In [ ]:
# Précision par classe (trier par les moins bonnes)
report_dict = {}
for cls in le.classes_:
    mask = y_test == le.transform([cls])[0]
    if mask.sum() == 0:
        continue
    acc_cls = accuracy_score(y_test[mask], y_pred_rf[mask])
    report_dict[cls] = {'accuracy': acc_cls, 'n_samples': int(mask.sum())}

df_cls = pd.DataFrame(report_dict).T.sort_values('accuracy')

fig, ax = plt.subplots(figsize=(14, 5))
colors_cls = [ACCENT if a >= 0.90 else '#FCD116' if a >= 0.75 else '#CE1126'
              for a in df_cls['accuracy']]
ax.bar(df_cls.index, df_cls['accuracy'], color=colors_cls, alpha=0.85, width=0.6)
ax.axhline(0.90, color='#C9D1D9', linestyle='--', linewidth=0.8, label='90 %')
ax.axhline(0.75, color='#FCD116', linestyle='--', linewidth=0.8, label='75 %')
ax.set_ylim(0, 1.05)
ax.set_title('Accuracy par classe  (rouge < 75 %, jaune < 90 %, vert ≥ 90 %)',
             color='#C9D1D9', fontsize=12)
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nClasses les moins bien reconnues :')
print(df_cls.head(5).to_string())

---
## 10. Résumé & Prochaines étapes

Exécute cette cellule en dernier pour afficher un bilan complet.

In [ ]:
with open(MODEL_DIR / 'model_info.json', 'r') as f:
    saved_meta = json.load(f)

print('╔══════════════════════════════════════════════════════╗')
print('║         SIGNBRIDGE — BILAN ENTRAÎNEMENT              ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Date           : {saved_meta["trained_at"][:19]:<33} ║')
print(f'║  Source dataset : {saved_meta["dataset_source"]:<33} ║')
print(f'║  Samples total  : {saved_meta["n_samples"]:>6,}  (train={saved_meta["n_train"]:,} / test={saved_meta["n_test"]:,})  ║')
print(f'║  Classes        : {saved_meta["n_classes"]:<33} ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  RF Accuracy    : {saved_meta["rf_accuracy"]:.2%}  (CV={saved_meta["cv_accuracy"]:.2%} ± {saved_meta["cv_std"]:.2%})  ║')
if 'lstm_accuracy' in saved_meta:
    print(f'║  LSTM Accuracy  : {saved_meta["lstm_accuracy"]:.2%}  ({saved_meta["lstm_epochs"]} epochs)          ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  rf_model.pkl   : {str(MODEL_DIR / "rf_model.pkl"):<33} ║')
print('╚══════════════════════════════════════════════════════╝')
print()
print('Prochaines étapes :')
print('  1. Redémarrer le serveur Django : daphne -p 8000 config.asgi:application')
print('  2. Tester via l\'interface : http://localhost:8000/app/?mode=camera')
print('  3. Ou via l\'admin : http://localhost:8000/admin-panel/training/')